# 01 Extraction

This notebook loads the raw Olist CSV files, confirms dataset suitability for the capstone, and documents the base schema that downstream notebooks will use.


In [1]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().resolve().name == 'notebooks' else Path.cwd().resolve()
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'

raw_files = {
    'customers': 'olist_customers_dataset.csv',
    'geolocation': 'olist_geolocation_dataset.csv',
    'order_items': 'olist_order_items_dataset.csv',
    'payments': 'olist_order_payments_dataset.csv',
    'reviews': 'olist_order_reviews_dataset.csv',
    'orders': 'olist_orders_dataset.csv',
    'products': 'olist_products_dataset.csv',
    'sellers': 'olist_sellers_dataset.csv',
    'translation': 'product_category_name_translation.csv',
}

for name, filename in raw_files.items():
    path = RAW_DIR / filename
    print(f'{name:12} -> {path.exists()} -> {filename}')


customers    -> True -> olist_customers_dataset.csv
geolocation  -> True -> olist_geolocation_dataset.csv
order_items  -> True -> olist_order_items_dataset.csv
payments     -> True -> olist_order_payments_dataset.csv
reviews      -> True -> olist_order_reviews_dataset.csv
orders       -> True -> olist_orders_dataset.csv
products     -> True -> olist_products_dataset.csv
sellers      -> True -> olist_sellers_dataset.csv
translation  -> True -> product_category_name_translation.csv


In [2]:
# Load each raw table.
dfs = {name: pd.read_csv(RAW_DIR / filename) for name, filename in raw_files.items()}

summary_rows = []
for name, df in dfs.items():
    summary_rows.append({
        'table_name': name,
        'rows': len(df),
        'columns': len(df.columns),
        'missing_cells': int(df.isna().sum().sum()),
        'duplicate_rows': int(df.duplicated().sum()),
    })

summary_df = pd.DataFrame(summary_rows).sort_values('rows', ascending=False)
summary_df


,table_name,rows,columns,missing_cells,duplicate_rows
1,geolocation,1000163,5,0,261831
2,order_items,112650,7,0,0
3,payments,103886,5,0,0
0,customers,99441,5,0,0
5,orders,99441,8,4908,0
4,reviews,99224,7,145903,0
6,products,32951,9,2448,0
7,sellers,3095,4,0,0
8,translation,71,2,0,0


In [3]:
# Inspect column structures.
for name, df in dfs.items():
    print(f'=== {name.upper()} ===')
    print(df.columns.tolist())


=== CUSTOMERS ===
['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']
=== GEOLOCATION ===
['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_state']
=== ORDER_ITEMS ===
['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']
=== PAYMENTS ===
['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']
=== REVIEWS ===
['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']
=== ORDERS ===
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']
=== PRODUCTS ===
['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g'

In [4]:
# Missing-value summary for the most important files.
for name in ['orders', 'reviews', 'products']:
    df = dfs[name]
    missing = df.isna().sum()
    missing = missing[missing > 0].sort_values(ascending=False)
    print(f'=== Missing values: {name} ===')
    print(missing if not missing.empty else 'No missing values')


=== Missing values: orders ===
order_delivered_customer_date    2965
order_delivered_carrier_date     1783
order_approved_at                 160
dtype: int64
=== Missing values: reviews ===
review_comment_title      87656
review_comment_message    58247
dtype: int64
=== Missing values: products ===
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64


In [5]:
# Show a few representative raw rows to understand granularity.
for name in ['orders', 'order_items', 'payments', 'reviews', 'products']:
    print(f'=== Sample rows: {name} ===')
    display(dfs[name].head(3))


=== Sample rows: orders ===


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00


=== Sample rows: order_items ===


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.9,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.9,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.0,17.87


=== Sample rows: payments ===


,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71


=== Sample rows: reviews ===


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24


=== Sample rows: products ===


,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0


## Extraction Notes

- The dataset comfortably exceeds the capstone minimum size requirements.
- It is a relational dataset, so ETL must handle one-to-many joins carefully to avoid duplicated orders in Tableau.
- Real missing values exist in `orders`, `reviews`, and `products`, which supports the data-cleaning requirement.
- The geolocation file is large and contains repeated zip prefixes, so it should be aggregated before being joined to customers or sellers.
